# LangChain + Groq Chatbot with Gradio

A simple ChatGPT-style chatbot built with:
- **LangChain** (latest, `langchain-groq` partner package)
- **Groq API** (fast LPU inference, e.g. Llama 3.3 models) via `ChatGroq`
- **Gradio** for the chat UI
- Conversation memory that keeps track of the **system message** and all **human/AI messages** across turns

> Paste your Groq API key directly in the code cell below marked `GROQ_API_KEY = "..."`.
> Get a free key from https://console.groq.com/keys

## 1. Install dependencies
Installs the latest LangChain core, the Groq partner package (`langchain-groq`), and Gradio.

In [3]:
!pip install -q -U langchain langchain-core langchain-groq gradio

## 2. Imports

In [4]:
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

print("Gradio version:", gr.__version__)

c:\Users\SAHANA\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio version: 6.22.0


## 3. Set your Groq API key
Paste your API key below between the quotes. Get one for free from https://console.groq.com/keys

In [ ]:
import os, getpass

GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Groq API key: ")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.7,
)

## 4. Conversation memory
We keep a single running list of LangChain message objects: one `SystemMessage` at the start, followed by alternating `HumanMessage` / `AIMessage` turns. This list is what gets sent to the model on every call, so it always has the full context of the conversation.

In [6]:
SYSTEM_PROMPT = "You are a helpful, friendly AI assistant, similar to ChatGPT. Answer clearly and concisely."

# The full conversation state (system + human + ai messages) lives here.
conversation_history = [SystemMessage(content=SYSTEM_PROMPT)]


def reset_conversation():
    """Reset the conversation back to just the system message."""
    global conversation_history
    conversation_history = [SystemMessage(content=SYSTEM_PROMPT)]

## 5. Chat function
This function appends the new human message to `conversation_history`, calls Groq with the full history (so the system message + prior turns are preserved), then appends the AI reply back into history. Any API error is caught and shown directly in the chat window instead of failing silently.

In [8]:
def chat_fn(message, history):
    """
    message: latest user message (str)
    history: Gradio's own chat history (list of dicts) - we ignore it for the
             actual LLM call and instead rely on our own `conversation_history`
             list of LangChain message objects, which is the source of truth.
    """
    global conversation_history

    # 1. add the new human message to the running conversation
    conversation_history.append(HumanMessage(content=message))

    try:
        # 2. call Groq with the full message list (system + all prior turns)
        ai_response = llm.invoke(conversation_history)
        reply_text = ai_response.content
    except Exception as e:
        # Show the real error in the chat window instead of failing silently
        reply_text = f"\u26a0\ufe0f Error calling Groq API: {type(e).__name__}: {e}"
        print(reply_text)  # also print full error to the notebook cell output
        # remove the human message we added since the call failed, so it can be retried cleanly
        conversation_history.pop()
        return reply_text

    # 3. store the AI reply back into history so future turns remember it
    conversation_history.append(AIMessage(content=reply_text))

    return reply_text

## 6. Gradio ChatGPT-style UI

In [9]:
def clear_fn():
    reset_conversation()
    return None


custom_css = """
@keyframes fadeInDown {
    from {opacity: 0; transform: translateY(-20px);}
    to {opacity: 1; transform: translateY(0);}
}
@keyframes gradientShift {
    0% {background-position: 0% 50%;}
    50% {background-position: 100% 50%;}
    100% {background-position: 0% 50%;}
}
@keyframes pulseGlow {
    0%, 100% {box-shadow: 0 0 8px rgba(167, 139, 250, 0.4);}
    50% {box-shadow: 0 0 20px rgba(244, 114, 182, 0.7);}
}
@keyframes twinkle {
    0% {opacity: 0; transform: scale(0) rotate(0deg);}
    40% {opacity: 1; transform: scale(1.2) rotate(90deg);}
    100% {opacity: 0; transform: scale(0.2) translateY(-60px) rotate(180deg);}
}

.gradio-container {
    background: linear-gradient(-45deg, #fdf2f8, #f5f3ff, #eff6ff, #fdf2f8);
    background-size: 400% 400%;
    animation: gradientShift 15s ease infinite;
}

#component-0 {
    animation: fadeInDown 0.8s ease-out;
}

button {
    transition: transform 0.2s ease, box-shadow 0.2s ease !important;
}
button:hover {
    transform: scale(1.05);
    animation: pulseGlow 1.5s infinite;
}

h1, .prose h1 {
    animation: fadeInDown 1s ease-out;
    background: linear-gradient(90deg, #a78bfa, #f472b6, #a78bfa);
    background-size: 200% auto;
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    animation: gradientShift 3s linear infinite;
}

/* Chatbot conversation panel background - nearly fully transparent */
#component-0 .block,
.chatbot,
[class*="chatbot"] {
    background: transparent !important;
    box-shadow: none !important;
}

/* ANY message bubble - fallback */
.message, .message-wrap, .bubble-wrap, [class*="message"] {
    animation: fadeInDown 0.4s ease-out;
}

/* USER's own message bubble (the "hello" box) -> pink, near-fully transparent */
.message.user,
.message-wrap .user,
[class*="user"] .message,
.chatbot [class*="user"],
[data-testid*="user"] {
    background: rgba(244, 114, 182, 0.04) !important;
    border: 1px solid rgba(244, 114, 182, 0.2) !important;
}

/* BOT's message bubble -> light purple, near-fully transparent */
.message.bot,
.message-wrap .bot,
[class*="bot"] .message,
.chatbot [class*="bot"] {
    background: rgba(196, 181, 253, 0.04) !important;
    border: 1px solid rgba(167, 139, 250, 0.2) !important;
}

/* Message input box: pink, near-fully transparent */
textarea, input[type="text"] {
    background: rgba(244, 114, 182, 0.03) !important;
    border: 1px solid rgba(244, 114, 182, 0.2) !important;
}

/* Flickering star elements spawned on send */
.spark-star {
    position: fixed;
    pointer-events: none;
    z-index: 999999;
    font-size: 22px;
    color: #f472b6;
    text-shadow: 0 0 6px #a78bfa, 0 0 10px #f9a8d4;
    animation: twinkle 900ms ease-out forwards;
}
"""

star_script = """
<img src="x" onerror="
(function(){
    if (window.__sparkStarsInit) return;
    window.__sparkStarsInit = true;

    function spawnSparkStars() {
        var colors = ['\\u2728', '\\u2b50', '\\u2733'];
        for (var i = 0; i < 16; i++) {
            var star = document.createElement('div');
            star.className = 'spark-star';
            star.textContent = colors[Math.floor(Math.random() * colors.length)];
            star.style.left = (Math.random() * window.innerWidth) + 'px';
            star.style.top = (window.innerHeight * (0.5 + Math.random() * 0.35)) + 'px';
            star.style.animationDelay = (Math.random() * 150) + 'ms';
            document.body.appendChild(star);
            (function(s){ setTimeout(function(){ s.remove(); }, 1200); })(star);
        }
    }

    function attachSparkListeners() {
        document.querySelectorAll('textarea, input[type=\\'text\\']').forEach(function(el){
            if (el.dataset.sparkBound) return;
            el.dataset.sparkBound = 'true';
            el.addEventListener('keydown', function(e){
                if (e.key === 'Enter' && !e.shiftKey) spawnSparkStars();
            });
        });
        document.querySelectorAll('button').forEach(function(btn){
            if (btn.dataset.sparkBound) return;
            var label = (btn.getAttribute('aria-label') || '') + ' ' + (btn.title || '');
            if (btn.querySelector('svg') || /send|submit/i.test(label)) {
                btn.dataset.sparkBound = 'true';
                btn.addEventListener('click', spawnSparkStars);
            }
        });
    }

    var sparkObserver = new MutationObserver(attachSparkListeners);
    sparkObserver.observe(document.body, {childList: true, subtree: true});
    attachSparkListeners();
    window.__spawnSparkStars = spawnSparkStars;
})();
" style="display:none;">
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# ✨ Spark AI")
    gr.Markdown("my Spark AI Chatbot")
    gr.ChatInterface(
        
        fn=chat_fn,
        examples=[
            "Hello, who are you?",
            "Explain quantum computing simply.",
            "Write a haiku about the ocean.",
        ],
    )
    gr.HTML(star_script)

# clicking the built-in "Clear" button in the UI also resets our LangChain history
demo.clear_fn = clear_fn

C:\Users\SAHANA\AppData\Local\Temp\ipykernel_22332\4179086877.py:149: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


## 7. Launch the app

In [10]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
